# 01 Finance Data Engineering — Ingestion Pipeline

## Project Context

This notebook implements Phase 0 of the Finance Data Engineering Stack — a portfolio project demonstrating a production-style market data ingestion and transformation pipeline in Python.

**Disclaimer:** This is a portfolio data engineering project. Nothing here constitutes investment advice or a production trading system.

## Ingestion Objective

1. Download adjusted close prices for a defined asset universe via yfinance.
2. Save raw OHLCV data for auditability.
3. Extract and clean the adjusted close price panel.
4. Compute daily simple returns.
5. Build SQL-ready dimension and fact tables.
6. Log the ingestion run metadata.
7. Save all outputs to `data/processed/` and `outputs/`.

## Asset Universe

| Ticker | Sector |
|---|---|
| AAPL | Technology |
| MSFT | Technology |
| NVDA | Technology |
| JPM  | Financials |
| PG   | Consumer Staples |
| KO   | Consumer Staples |
| XOM  | Energy |
| JNJ  | Health Care |
| SPY  | ETF (S&P 500) |

In [ ]:
import os
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

# Resolve project root from notebook CWD (nbconvert sets CWD = notebook dir)
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(5):
    if (PROJECT_ROOT / "src").is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RAW_DATA_DIR, PROCESSED_DATA_DIR,
    LOGS_DIR, SUMMARIES_DIR,
    ASSET_UNIVERSE, DEFAULT_START_DATE,
)
from src.ingestion import (
    download_market_data, extract_adjusted_close,
    save_raw_market_data, save_processed_market_data,
    create_ingestion_log,
)
from src.transforms import (
    calculate_returns, create_asset_dimension,
    create_price_fact_table, create_returns_fact_table,
)

TICKERS    = ASSET_UNIVERSE
START_DATE = DEFAULT_START_DATE
RUN_TS     = datetime.now(tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

print(f"Project root : {PROJECT_ROOT}")
print(f"Tickers      : {TICKERS}")
print(f"Start date   : {START_DATE}")
print(f"Run timestamp: {RUN_TS}")

## Download Market Data (yfinance)

In [ ]:
raw_data = download_market_data(TICKERS, start_date=START_DATE)

print(f"Raw data shape : {raw_data.shape}")
print(f"Date range     : {raw_data.index.min().date()} to {raw_data.index.max().date()}")
print(f"Columns (first 10): {list(raw_data.columns[:10])}")

## Save Raw Market Data

In [ ]:
raw_path = RAW_DATA_DIR / "market_prices_raw.csv"
save_raw_market_data(raw_data, raw_path)
print(f"Raw data saved: {raw_path.name}  ({raw_path.stat().st_size:,} bytes)")

## Extract Adjusted Close Prices

In [ ]:
prices = extract_adjusted_close(raw_data)

print(f"Prices shape : {prices.shape}")
print(f"Tickers      : {list(prices.columns)}")
print(f"Date range   : {prices.index.min().date()} to {prices.index.max().date()}")
print(f"\nLatest prices:")
print(prices.tail(3).round(2).to_string())

## Save Processed Adjusted Close Prices

In [ ]:
prices_path = PROCESSED_DATA_DIR / "adjusted_close_prices.csv"
save_processed_market_data(prices, prices_path)
print(f"Saved: {prices_path.name}  ({prices_path.stat().st_size:,} bytes)")

## Compute Daily Returns

In [ ]:
returns = calculate_returns(prices)

returns_path = PROCESSED_DATA_DIR / "daily_returns.csv"
save_processed_market_data(returns, returns_path)

print(f"Returns shape: {returns.shape}")
print(f"Saved       : {returns_path.name}  ({returns_path.stat().st_size:,} bytes)")
print(f"\nSample returns (last 3 rows):")
print(returns.tail(3).round(5).to_string())

## Build Asset Dimension Table

In [ ]:
dim_assets = create_asset_dimension(TICKERS)

dim_path = PROCESSED_DATA_DIR / "dim_assets.csv"
dim_assets.to_csv(dim_path, index=False)

print(f"dim_assets rows : {len(dim_assets)}")
print(f"Saved           : {dim_path.name}")
print()
print(dim_assets.to_string(index=False))

## Build Price Fact Table

In [ ]:
fact_prices = create_price_fact_table(prices)

fp_path = PROCESSED_DATA_DIR / "fact_prices.csv"
fact_prices.to_csv(fp_path, index=False)

print(f"fact_prices rows: {len(fact_prices):,}")
print(f"Columns         : {list(fact_prices.columns)}")
print(f"Saved           : {fp_path.name}  ({fp_path.stat().st_size:,} bytes)")
print()
print(fact_prices.tail(5).to_string(index=False))

## Build Returns Fact Table

In [ ]:
fact_returns = create_returns_fact_table(returns)

fr_path = PROCESSED_DATA_DIR / "fact_returns.csv"
fact_returns.to_csv(fr_path, index=False)

print(f"fact_returns rows: {len(fact_returns):,}")
print(f"Columns          : {list(fact_returns.columns)}")
print(f"Saved            : {fr_path.name}  ({fr_path.stat().st_size:,} bytes)")
print()
print(fact_returns.tail(5).to_string(index=False))

## Build Ingestion Log

In [ ]:
log_records = []
for ticker in TICKERS:
    col_data = prices[ticker].dropna() if ticker in prices.columns else pd.Series(dtype=float)
    log_records.append({
        "ticker":          ticker,
        "start_date":      START_DATE,
        "end_date":        str(prices.index.max().date()),
        "rows_downloaded": len(col_data),
        "missing_count":   int(prices[ticker].isnull().sum()) if ticker in prices.columns else -1,
        "latest_price":    round(float(col_data.iloc[-1]), 4) if not col_data.empty else None,
        "status":          "ok" if not col_data.empty else "no_data",
        "run_timestamp":   RUN_TS,
    })

log_path = LOGS_DIR / "ingestion_log.csv"
log_df = create_ingestion_log(log_records, log_path)

print(f"Ingestion log saved: {log_path.name}")
print(log_df.to_string(index=False))

## Data Summary

In [ ]:
# Descriptive stats on adjusted close prices
print("=== Adjusted Close Price Summary ===")
print(prices.describe().round(2).to_string())

print()
print("=== Daily Return Summary (%) ===")
print((returns * 100).describe().round(4).to_string())

# Missing value check
print()
print("=== Missing Value Counts ===")
print(prices.isnull().sum().to_string())

In [ ]:
# Save ingestion summary
summary_records = [{
    "run_timestamp":      RUN_TS,
    "tickers_requested":  len(TICKERS),
    "tickers_ingested":   int((log_df["status"] == "ok").sum()),
    "date_range_start":   START_DATE,
    "date_range_end":     str(prices.index.max().date()),
    "trading_days":       len(prices),
    "total_price_rows":   len(fact_prices),
    "total_return_rows":  len(fact_returns),
    "raw_file":           str(raw_path.name),
    "prices_file":        str(prices_path.name),
    "returns_file":       str(returns_path.name),
    "dim_assets_file":    str(dim_path.name),
    "fact_prices_file":   str(fp_path.name),
    "fact_returns_file":  str(fr_path.name),
}]

summary_df = pd.DataFrame(summary_records)
summary_path = SUMMARIES_DIR / "ingestion_summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"Summary saved: {summary_path.name}")
print(summary_df.T.to_string(header=False))

## Output Verification

In [ ]:
output_files = [
    raw_path,
    prices_path,
    returns_path,
    dim_path,
    fp_path,
    fr_path,
    log_path,
    summary_path,
]

print("Phase 0 output verification:")
print("-" * 65)
all_ok = True
for p in output_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if (exists and size > 0) else "MISSING"
    if status != "OK":
        all_ok = False
    rel = p.relative_to(PROJECT_ROOT)
    print(f"  {status:<8} {size:>10,} bytes  {rel}")
print("-" * 65)
print("All outputs verified." if all_ok else "WARNING: one or more outputs missing.")

## Limitations

- **yfinance dependency:** Data depends on Yahoo Finance availability. If the service is unavailable or rate-limits the request, ingestion fails. A CSV fallback is a Phase 1 improvement.
- **No validation yet:** Missing values, stale prices, and return outliers are not checked here. Phase 1 implements the full validation layer.
- **No warehouse yet:** Outputs are CSV files. DuckDB population and SQL querying are Phase 1 deliverables.
- **Batch ingestion only:** The pipeline downloads the full history on each run. Incremental ingestion is a future improvement.
- **Adjusted close only:** OHLCV data is saved in raw form but only adjusted close is used in processed tables. Volume and open/high/low are Phase 1 candidates.

## Next Steps for Phase 1

1. Implement data quality checks in `src/validation.py` — missing values, price staleness, return outliers.
2. Populate DuckDB warehouse from the fact/dim CSVs.
3. Run `sql/sample_queries.sql` against the warehouse.
4. Build `notebooks/02_data_quality_and_warehouse.ipynb`.
5. Generate `reports/data_quality_report.md`.